# Generate dataset with random policy

In [1]:
import torch
from tqdm import tqdm
from data_collector import wrapper_collector


device = "cuda:0" if torch.cuda.is_available() else "cpu"
seed = 0
num_envs = 1
action_repeat = 1

In [ ]:
env_name = "Go1JoystickFlatTerrain"
env = wrapper_collector(env_name, num_envs, seed, action_repeat, device)

In [3]:
import minari

dataset_id = str("playground/" + env_name + "-v0")

# delete the test dataset if it already exists
local_datasets = minari.list_local_datasets()
if dataset_id in local_datasets:
    minari.delete_dataset(dataset_id)

In [ ]:
num_samples = 1000

env.reset()
timesteps = 0
pbar = tqdm(total=num_samples, desc="Collecting samples")
while timesteps < num_samples:
    terminated = torch.zeros(num_envs, device=device).bool()
    truncated = torch.zeros(num_envs, device=device).bool()
    while not terminated.all():
        action = env.action_sample(num_envs)
        _, _, terminated, truncated, _, _ = env.step(action)
        pbar.update(env.get_timesteps() - timesteps)
        timesteps = env.get_timesteps()
    env.reset()
pbar.close()

In [ ]:
dataset = env.create_dataset(
    dataset_id=str("playground/" + env_name + "-v0"),
    algorithm_name="random_policy",
    author="Luana Martins",
    author_email="luanagbmartins@gmail.com",
    code_permalink="https://github.com/AKCIT-RL/mujoco_playground",
    description="Random policy for " + env_name + " of mujoco_playground",
)

In [ ]:
dataset.total_episodes, dataset.total_steps